# LeetCode #1123 — Lowest Common Ancestor of Deepest Leaves

**Medium · Python 3 · Binary Tree, BFS, DFS, Tree Dynamic Programming**  
[Đề gốc](https://leetcode.com/problems/lowest-common-ancestor-of-deepest-leaves/) · [Mô tả và ràng buộc](https://leetcode.doocs.org/en/lc/1123/)

```python
class Solution:
    def lcaDeepestLeaves(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        ...
```

Ta sẽ tự tìm các lá sâu nhất, tìm **tổ tiên chung gần nhất** (LCA), rồi rút ra lời giải **DFS hậu tự trả về hai thông tin**: độ sâu và nút ứng viên.

## 1. Mục tiêu học

1. Phân biệt **lá**, **độ sâu từ gốc**, **chiều cao cây con** và **tổ tiên**.
2. Hiểu LCA của **một tập nhiều lá**, không chỉ hai nút.
3. Tự giải bằng **BFS + đường đi** như cách trực giác.
4. Rút ra công thức DFS: so chiều cao hai cây con để chọn ứng viên.
5. Viết mã xử lý cây lệch sâu đến 1000 nút bằng **DFS hậu tự không đệ quy**.

## 2. Đề, ví dụ và giới hạn

Cho một cây nhị phân có gốc `root`. **Lá** là nút không có con. Lá sâu nhất là lá có khoảng cách từ gốc lớn nhất. Trả về **nút tổ tiên chung thấp nhất (LCA) của tất cả các lá sâu nhất**.

- Ví dụ 1: `root=[3,5,1,6,2,0,8,None,None,7,4]` → trả về **nút 2** (cây con `[2,7,4]`).
- Ví dụ 2: `root=[1]` → nút **1**.
- Ví dụ 3: `root=[0,1,3,None,2]` → nút **2** (vì chỉ một lá sâu nhất).

**Chú ý:** Bài yêu cầu trả về **đối tượng `TreeNode`**, không phải số `node.val` hay list giá trị. Có 1–1000 nút, giá trị khác nhau thuộc `[0,1000]`. Đề lấy gốc ở độ sâu 0.

## 3. Kiến thức nền: nút, lá, độ sâu, chiều cao

- **Depth(node):** số cạnh từ gốc đến node; gốc có depth 0.
- **Height(node):** số nút trên đường đi dài nhất từ node xuống một lá; ở notebook này `height(None)=0`, `height(leaf)=1`.
- **LCA của nhiều nút:** nút ở vị trí thấp nhất mà cây con của nó chứa **tất cả** các nút đã chọn. Một nút được xem là tổ tiên của chính nó.

Đừng nhầm **lá sâu nhất từ gốc** và **lá sâu nhất trong một cây con**: DFS cần biết chiều cao tương đối để ghép kết quả ở cha.

In [1]:
from collections import deque
from typing import List, Optional

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f'TreeNode({self.val})'


def build_tree(values):
    """Dựng cây theo thứ tự tầng của LeetCode; None nghĩa là thiếu nút."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    pending = deque([root])
    i = 1
    while pending and i < len(values):
        node = pending.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            pending.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            pending.append(node.right)
        i += 1
    return root


def serialize_tree(root):
    """Đổi cây thành list theo tầng, loại bỏ None dư ở cuối."""
    if root is None:
        return []
    result, pending = [], deque([root])
    while pending:
        node = pending.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        pending.extend([node.left, node.right])
    while result and result[-1] is None:
        result.pop()
    return result


def forest_as_sorted_lists(roots):
    """So sánh rừng mà không phụ thuộc thứ tự các cây được trả về."""
    return sorted((serialize_tree(root) for root in roots), key=str)

### 3.1. Minh họa cây mẫu

```text
         3 (depth 0)
        / \
       5   1 (depth 1)
      / \ / \
     6  2 0  8 (depth 2)
       / \
      7   4 (depth 3, hai lá sâu nhất)
```

Hai lá `7` và `4` có LCA là `2`, **không phải gốc 3**. Vì chỉ cần tổ tiên chung thấp nhất, ta chọn nút 2.

In [2]:
example1=build_tree([3,5,1,6,2,0,8,None,None,7,4])
print('Cây ví dụ:',serialize_tree(example1))
print('Gốc:',example1.val,'nút 2:',example1.left.right.val)

Cây ví dụ: [3, 5, 1, 6, 2, 0, 8, None, None, 7, 4]
Gốc: 3 nút 2: 2


## 4. Python nền tảng: `deque` để BFS và tuple `(node,depth)`

BFS (breadth-first search) duyệt theo tầng: `deque.popleft()` lấy nút ở đầu hàng chờ; `append()` đưa các con vào cuối. Một **tuple** như `(node, depth)` lưu đồng thời nút và khoảng cách hiện tại. Tránh dùng `list.pop(0)` vì dịch các phần tử, tốn O(n) mỗi lần.

In [3]:
def bfs_nodes_with_depth(root):
    if root is None: return []
    q=deque([(root,0)])
    order=[]
    while q:
        node,depth=q.popleft()
        order.append((node.val,depth))
        if node.left: q.append((node.left,depth+1))
        if node.right: q.append((node.right,depth+1))
    return order

print(bfs_nodes_with_depth(build_tree([3,5,1,6,2,0,8,None,None,7,4])))

[(3, 0), (5, 1), (1, 1), (6, 2), (2, 2), (0, 2), (8, 2), (7, 3), (4, 3)]


## 5. Cách dễ nghĩ nhất (baseline): ghi toàn bộ đường đi tới các lá

- BFS đến từng lá; lưu đường đi `[root,...,leaf]`.
- Chọn những đường đi có **độ dài lớn nhất** (lá sâu nhất).
- Xem **phần đầu chung dài nhất** của mọi đường đi; phần tử cuối của phần đầu chung là LCA.

Đơn giản để giải thích, nhưng việc **sao chép đường đi** ở mỗi bước có thể tốn tới O(nh) thời gian/bộ nhớ, với h là chiều cao (worst-case O(n²)).

In [4]:
def lca_by_paths(root):
    if root is None: return None
    q=deque([(root,[root])])
    deepest_paths=[]; maximum=0
    while q:
        node,path=q.popleft()
        if not node.left and not node.right:
            if len(path)>maximum:
                maximum=len(path)
                deepest_paths=[path]
            elif len(path)==maximum:
                deepest_paths.append(path)
        if node.left: q.append((node.left,path+[node.left]))
        if node.right: q.append((node.right,path+[node.right]))
    answer=deepest_paths[0][0]
    for i in range(maximum):
        candidate=deepest_paths[0][i]
        if all(path[i] is candidate for path in deepest_paths):
            answer=candidate
        else:
            break
    return answer

for values in ([3,5,1,6,2,0,8,None,None,7,4],[1],[0,1,3,None,2]):
    print(values,'-> LCA=',lca_by_paths(build_tree(values)).val)

[3, 5, 1, 6, 2, 0, 8, None, None, 7, 4] -> LCA= 2
[1] -> LCA= 1
[0, 1, 3, None, 2] -> LCA= 2


### 5.1. Tại sao so nút bằng `is` thay vì so `val`?

Bài trả về **đối tượng** nút. `node1 is node2` kiểm tra hai biến có trỏ tới **cùng nút trong cây** không. Vì đề đảm bảo giá trị khác nhau, so `val` cũng phân biệt được ở đây, nhưng cách nghĩ **theo danh tính nút** đúng ngữ nghĩa LCA và mở rộng cho cây có giá trị trùng.

In [5]:
a=TreeNode(2)
b=TreeNode(2)
print('a.val == b.val:',a.val==b.val)
print('a is b:',a is b)
assert a.val==b.val and a is not b

a.val == b.val: True
a is b: False


## 6. Bước đột phá: mỗi cây con chỉ cần trả 2 thông tin

Với một nút `node`, giả sử đã giải xong hai cây con:
- Trái trả `(heightL, candidateL)`.
- Phải trả `(heightR, candidateR)`.

**Ba trường hợp:**

1. `heightL > heightR`: lá sâu nhất trong **cây con hiện tại** đều nằm bên trái → giữ `candidateL`.
2. `heightL < heightR`: tất cả nằm bên phải → giữ `candidateR`.
3. `heightL == heightR`: cả hai phía có lá đạt chiều sâu bằng nhau → **nút hiện tại chính là LCA** của các lá sâu nhất.

Chiều cao trả lên luôn là `max(heightL,heightR)+1`. Với `None` trả `(0,None)`.

In [6]:
# Một bảng quyết định mini, trước khi viết đệ quy:
for left_height,right_height in [(3,1),(1,3),(2,2),(0,0)]:
    if left_height>right_height: choice='ứng viên TRÁI'
    elif right_height>left_height: choice='ứng viên PHẢI'
    else: choice='NÚT HIỆN TẠI'
    print(f'heightL={left_height}, heightR={right_height} → {choice}, '
          f'height trả về={max(left_height,right_height)+1}')

heightL=3, heightR=1 → ứng viên TRÁI, height trả về=4
heightL=1, heightR=3 → ứng viên PHẢI, height trả về=4
heightL=2, heightR=2 → NÚT HIỆN TẠI, height trả về=3
heightL=0, heightR=0 → NÚT HIỆN TẠI, height trả về=1


## 7. Nền tảng đệ quy: base case và postorder

**Đệ quy** là hàm gọi lại chính mình trên bài nhỏ hơn. Ở cây:
1. `node is None` → trả cặp `(0,None)` (base case).
2. Hỏi kết quả của cây con trái/phải.
3. Dùng chiều cao của con để quyết định ứng viên cho cha.

Đây là **postorder (trái → phải → nút)**: chỉ kết luận về node sau khi có đầy đủ kết quả của các con.

In [7]:
def subtree_result_recursive(node):
    if node is None:
        return 0,None
    height_left,ancestor_left=subtree_result_recursive(node.left)
    height_right,ancestor_right=subtree_result_recursive(node.right)
    height=1+max(height_left,height_right)
    if height_left==height_right:
        return height,node
    if height_left>height_right:
        return height,ancestor_left
    return height,ancestor_right

height,answer=subtree_result_recursive(build_tree([3,5,1,6,2,0,8,None,None,7,4]))
print('Chiều cao=',height,'; LCA sâu nhất=',answer.val)
assert height==4 and answer.val==2

Chiều cao= 4 ; LCA sâu nhất= 2


## 8. Trace từ lá đi lên với cây ví dụ

| Nút | `heightL` | `heightR` | Ứng viên trả lên | `height` |
|---:|---:|---:|---:|---:|
| 7 | 0 | 0 | 7 | 1 |
| 4 | 0 | 0 | 4 | 1 |
| **2** | 1 | 1 | **2** | 2 |
| 5 | 1 (nút 6) | 2 (nút 2) | 2 | 3 |
| 1 | 1 (nút 0) | 1 (nút 8) | 1 | 2 |
| **3** | 3 (nút 5) | 2 (nút 1) | **2** | 4 |

LCA tại nút 2 vì cây con bên trái và bên phải của 2 cùng có một lá sâu nhất (7 và 4). Nút 3 không chọn chính nó vì nhánh bên trái sâu hơn nhánh phải.

In [8]:
def trace_lca(node):
    def visit(cur):
        if cur is None: return 0,None
        left_depth,left_lca=visit(cur.left)
        right_depth,right_lca=visit(cur.right)
        height=1+max(left_depth,right_depth)
        if left_depth==right_depth: lca=cur
        elif left_depth>right_depth:lca=left_lca
        else:lca=right_lca
        print(f'nút={cur.val:>2} | L={left_depth} R={right_depth} '
              f'| trả height={height}, LCA={lca.val}')
        return height,lca
    return visit(node)

trace_lca(build_tree([3,5,1,6,2,0,8,None,None,7,4]))

nút= 6 | L=0 R=0 | trả height=1, LCA=6
nút= 7 | L=0 R=0 | trả height=1, LCA=7
nút= 4 | L=0 R=0 | trả height=1, LCA=4
nút= 2 | L=1 R=1 | trả height=2, LCA=2
nút= 5 | L=1 R=2 | trả height=3, LCA=2
nút= 0 | L=0 R=0 | trả height=1, LCA=0
nút= 8 | L=0 R=0 | trả height=1, LCA=8
nút= 1 | L=1 R=1 | trả height=2, LCA=1
nút= 3 | L=3 R=2 | trả height=4, LCA=2


(4, TreeNode(2))

## 9. Nhánh rất sâu: tránh `RecursionError`

Với tối đa 1000 nút, có thể gặp cây **lệch 1000 nút**. DFS đệ quy chuẩn có nguy cơ đụng giới hạn call stack của Python. Ta có thể mô phỏng postorder bằng **stack tường minh**.

`stack` lưu `(node,processed)`:
- `processed=False`: lần đầu thấy node → xếp lại node với True rồi xếp các con để xử lý trước.
- `processed=True`: hai con đã có kết quả trong `memo` → tính `(height, LCA)` của node.

`memo[node]` là dictionary dùng **đối tượng nút** làm khóa; vì `TreeNode` giữ identity mặc định, mỗi nút là một khóa riêng.

In [9]:
def subtree_result_iterative(root):
    if root is None: return 0,None
    memo={}
    stack=[(root,False)]
    while stack:
        node,processed=stack.pop()
        if node is None: continue
        if not processed:
            stack.append((node,True))
            stack.append((node.right,False))
            stack.append((node.left,False))
        else:
            lh,llca=memo.get(node.left,(0,None))
            rh,rlca=memo.get(node.right,(0,None))
            height=1+max(lh,rh)
            if lh==rh: candidate=node
            elif lh>rh: candidate=llca
            else: candidate=rlca
            memo[node]=(height,candidate)
    return memo[root]

h,lca=subtree_result_iterative(build_tree([3,5,1,6,2,0,8,None,None,7,4]))
assert h==4 and lca.val==2
print('Iterative postorder: height',h,'LCA',lca.val)

Iterative postorder: height 4 LCA 2


### 9.1. Giải thích `memo.get(node.left, (0,None))`

- `memo`: dictionary lưu kết quả đã tính cho từng nút.
- `memo.get(key,default)` trả giá trị của `key`; nếu không tồn tại thì trả `default`.
- Với `node.left is None`, `None` không có trong memo → kết quả `(0,None)` đúng trường hợp cơ sở.

Nếu `node.left` là nút thật, nó đã được xử lý trước cha nên **chắc chắn** có trong memo.

In [10]:
memo={}
node=TreeNode(9)
print('Chưa có:',memo.get(node,(0,None)))
memo[node]=(1,node)
print('Sau khi ghi:',memo.get(node,(0,None)))
assert memo.get(None,(0,None))==(0,None)

Chưa có: (0, None)
Sau khi ghi: (1, TreeNode(9))


## 10. Code nộp LeetCode (bản an toàn với cây lệch sâu)

Code sau giữ đúng chữ ký và trả về **TreeNode**. `TreeNode` được LeetCode cung cấp nên trong trang nộp không cần paste code dựng cây hay `serialize_tree`.

In [11]:
class Solution:
    def lcaDeepestLeaves(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        if root is None:
            return None
        dp={}
        stack=[(root,False)]
        while stack:
            node,done=stack.pop()
            if node is None:
                continue
            if not done:
                stack.append((node,True))
                stack.append((node.right,False))
                stack.append((node.left,False))
                continue
            h_left,lca_left=dp.get(node.left,(0,None))
            h_right,lca_right=dp.get(node.right,(0,None))
            if h_left==h_right:
                chosen=node
            elif h_left>h_right:
                chosen=lca_left
            else:
                chosen=lca_right
            dp[node]=(1+max(h_left,h_right),chosen)
        return dp[root][1]

got=Solution().lcaDeepestLeaves(build_tree([3,5,1,6,2,0,8,None,None,7,4]))
print('Trả về đối tượng:',got,'; cây con:',serialize_tree(got))
assert serialize_tree(got)==[2,7,4]

Trả về đối tượng: TreeNode(2) ; cây con: [2, 7, 4]


## 11. Chứng minh bằng quy nạp trên chiều cao

**Mệnh đề**: với mỗi nút `v`, kết quả `(height,candidate)` là chiều cao cây con v và LCA của **tất cả lá sâu nhất bên trong cây con v**.

- **Base case:** `None` có height=0, candidate=None. Một lá có hai cây con rỗng → height=1, candidate là chính lá.
- **Bước quy nạp:** giả sử kết quả hai con đúng. Nếu nhánh trái cao hơn, lá sâu nhất ở v chỉ có thể thuộc trái, nên chọn ứng viên trái; tương tự phải. Nếu hai nhánh bằng chiều cao, cả hai phía đều chứa lá sâu nhất, nên tổ tiên chung thấp nhất của chúng là **v**.
- **Kết luận:** áp dụng ở `root`, ứng viên là LCA của các lá sâu nhất của cả cây.

Phiên bản iterative chỉ thay cách tổ chức thứ tự thực hiện postorder, không thay công thức nên giữ nguyên tính đúng.

## 12. Time/Space Complexity

Gọi `n` là số nút, `h` là chiều cao cây (số mức trên đường dài nhất).

| Phương pháp | Thời gian | Bộ nhớ thêm |
|---|---|---|
| BFS sao chép đường đi tới lá | O(nh) | O(nh) worst-case để giữ đường đi |
| DFS đệ quy `(height,LCA)` | **O(n)** | O(h) call stack |
| **Iterative postorder + memo** | **O(n)** | **O(n)** cho dictionary + stack |

`n<=1000`; bản iterative không bị giới hạn số lời gọi đệ quy nhưng dùng thêm O(n) bộ nhớ để lưu kết quả từng nút.

## 13. Test các ví dụ, cây cân bằng, lệch và một lá sâu nhất

**Quan trọng:** `lcaDeepestLeaves` trả **nút**, vì vậy `serialize_tree(got)` chỉ là cách kiểm tra cây con từ nút được trả về. Test cũng dùng `is` để bảo đảm hàm trả lại **chính nút có trong cây gốc**, không phải nút được tạo mới.

In [12]:
examples=[
    ([3,5,1,6,2,0,8,None,None,7,4],[2,7,4]),
    ([1],[1]),
    ([0,1,3,None,2],[2]),
    ([1,2,3],[1,2,3]),
    ([1,2,None,3,None,4],[4]),
    ([1,None,2,None,3],[3]),
    ([1,2,3,4,None,None,5],[1,2,3,4,None,None,5]),
]
for values,expected_subtree in examples:
    root=build_tree(values)
    got=Solution().lcaDeepestLeaves(root)
    assert serialize_tree(got)==expected_subtree,(values,serialize_tree(got))
print('PASS',len(examples),'ví dụ và trường hợp biên')

PASS 7 ví dụ và trường hợp biên


### 13.1. Oracle độc lập: BFS toàn bộ đường đi

So sánh **đối tượng** (`is`) giữa lời giải O(n) và `lca_by_paths` trên cây mới sinh, thay vì chỉ so giá trị. Oracle đi theo hướng BFS + các đường đi nên có cách suy nghĩ khác DFS DP.

In [13]:
import random
rng=random.Random(1123)
def random_tree(n):
    values=list(range(n))
    rng.shuffle(values)
    root=TreeNode(values[0]);slots=[(root,'left'),(root,'right')]
    for x in values[1:]:
        k=rng.randrange(len(slots))
        par,side=slots.pop(k)
        child=TreeNode(x);setattr(par,side,child)
        slots.extend([(child,'left'),(child,'right')])
    return root

for _ in range(250):
    root=random_tree(rng.randint(1,35))
    expected=lca_by_paths(root)
    actual=Solution().lcaDeepestLeaves(root)
    assert actual is expected,(serialize_tree(root),actual,expected)
print('PASS 250 cây ngẫu nhiên (so danh tính nút)')

PASS 250 cây ngẫu nhiên (so danh tính nút)


### 13.2. Stress test cây lệch 1000 nút

Với cây một nhánh dài 1000, nút duy nhất sâu nhất chính là **lá cuối**. Code không đệ quy phải trả lá đó mà không cần đổi giới hạn Python.

In [14]:
chain=TreeNode(1);cursor=chain
for value in range(2,1001):
    cursor.right=TreeNode(value)
    cursor=cursor.right
result=Solution().lcaDeepestLeaves(chain)
assert result is cursor and result.val==1000
print('PASS n=1000, cây lệch một phía; LCA=',result.val)

PASS n=1000, cây lệch một phía; LCA= 1000


## 14. Lỗi thường gặp (code sai → nguyên nhân → sửa)

1. **Trả về độ sâu thay vì nút:** đề yêu cầu `TreeNode`, không phải `int`.
2. **Gặp hai cây con bằng độ cao nhưng chọn trái:** sai vì cần LCA của lá sâu nhất trên **cả hai bên**; đúng là chọn node hiện tại.
3. **Nhầm depth và height:** depth tính từ gốc; height ở DFS tính từ node xuống lá và phải +1 khi lên cha.
4. **Chỉ lấy một lá sâu nhất:** có thể còn lá khác cùng độ sâu, khiến LCA phải cao hơn.
5. **So `val` thay vì node identity khi mở rộng bài:** hai nút khác nhau vẫn có thể cùng val ngoài ràng buộc của đề.
6. **Đệ quy trên cây lệch 1000 nút:** có thể chạm giới hạn stack của Python.

In [15]:
# Phản ví dụ nếu chỉ chọn một lá sâu nhất:
root=build_tree([1,2,3])
assert Solution().lcaDeepestLeaves(root) is root
print('Hai lá 2,3 cùng sâu -> LCA phải là gốc 1')

# Cây chỉ có một lá sâu nhất thì LCA chính là lá đó:
root=build_tree([1,2,3,4])
got=Solution().lcaDeepestLeaves(root)
assert got.val==4

Hai lá 2,3 cùng sâu -> LCA phải là gốc 1


## 15. Một cách nhìn khác: BFS + cha và gom tổ tiên

Có thể dùng BFS để tìm toàn bộ lá ở tầng sâu nhất, lưu `parent[child]=node`. Sau đó **đồng loạt đi lên cha** từ các nút sâu nhất, bỏ trùng bằng `set`, tới khi chỉ còn một nút. Đây là cách độc lập, không cần postorder và cũng tránh đệ quy.

In [16]:
def lca_by_parent_bfs(root):
    if root is None: return None
    parent={root:None};queue=deque([root]);last_level=[]
    while queue:
        last_level=[]
        for _ in range(len(queue)):
            node=queue.popleft()
            last_level.append(node)
            for child in (node.left,node.right):
                if child:
                    parent[child]=node
                    queue.append(child)
    candidates=set(last_level)
    while len(candidates)>1:
        candidates={parent[node] for node in candidates}
    return next(iter(candidates))

for values,_ in examples:
    root=build_tree(values)
    assert lca_by_parent_bfs(root) is Solution().lcaDeepestLeaves(root)
print('PASS giải pháp BFS cha')

PASS giải pháp BFS cha


## 16. Dấu hiệu nhận dạng kỹ thuật

- “Nút thấp nhất chứa **mọi lá** có thuộc tính tốt nhất” → nghĩ tới cây con và LCA.
- Kết quả của cha quyết định hoàn toàn từ hai con → **DFS postorder / tree DP**.
- Mỗi cây con trả **hơn một thông tin** → dùng tuple `(height, candidate)`.
- Cần so chiều cao hai nhánh rồi ưu tiên nhánh cao hơn → pattern thường gặp ở balanced tree, diameter, deepest subtree.

Biến thể: trả giá trị LCA; đếm số lá sâu nhất; trả một danh sách tất cả lá sâu nhất; tính chiều cao từ gốc.

In [17]:
def deepest_leaf_values(root):
    if root is None: return []
    queue=deque([root]);last=[]
    while queue:
        last=[]
        for _ in range(len(queue)):
            node=queue.popleft()
            last.append(node)
            if node.left:queue.append(node.left)
            if node.right:queue.append(node.right)
    return [node.val for node in last]

assert deepest_leaf_values(build_tree([3,5,1,6,2,0,8,None,None,7,4]))==[7,4]
print('Lá sâu nhất:',[7,4])

Lá sâu nhất: [7, 4]


## 17. Tự luyện (5 bài)

1. **Tính tay:** `root=[1,2,3,4,5]`; lá sâu nhất nào? LCA của chúng là gì?
2. **Dự đoán output:** Với `None`, `(0,None)` thể hiện điều gì trong thuật toán DFS?
3. **TODO:** viết `exercise_height(root)` tính chiều cao theo số **nút**.
4. **TODO:** viết `exercise_lca_recursive(root)` bằng cách trả `(height,lca)` từ mỗi cây con.
5. **Biến thể:** viết `exercise_deepest_count(root)` đếm số lá ở độ sâu tối đa của cây.

In [18]:
def exercise_height(root):
    # TODO: với None trả 0; còn lại 1 + max(chiều cao con trái/phải).
    pass

def exercise_lca_recursive(root):
    # TODO: dfs trả (height, ứng viên LCA) và so height hai nhánh.
    pass

def exercise_deepest_count(root):
    # TODO: BFS theo từng tầng và đếm nút tại tầng cuối.
    pass

# Tự làm xong mới bỏ #:
# assert exercise_height(build_tree([1,2,3,4]))==3
# assert exercise_lca_recursive(build_tree([1,2,3])) .val == 1
# assert exercise_deepest_count(build_tree([1,2,3,4,5]))==2

## 18. Gợi ý 3 mức

| Bài | Tư duy | Cú pháp cần | Gần-code |
|---|---|---|---|
| 3 | Height của None bằng 0 | `max` | `1 + max(height(left),height(right))` |
| 4 | Trả hai thông tin | `return height,node` | hai con bằng → chọn `node`; không bằng → chọn bên cao hơn |
| 5 | Mỗi vòng `while` xử lý một tầng | `deque`, `len(queue)` | sau khi BFS kết thúc, số nút tầng cuối là `len(last_level)` |

Không cần ghi nhớ code nguyên xi: tự chứng minh khi nào chọn node, khi nào kế thừa đáp án từ con.

## 19. Đáp án tham khảo — ở sau bài tập

**Bài 1:** lá sâu nhất là `4` và `5`, LCA là nút `2`.  
**Bài 2:** `None` không có nút, chiều cao bằng 0 và chưa có ứng viên LCA.

In [19]:
def answer_height(root):
    if root is None:return 0
    return 1+max(answer_height(root.left),answer_height(root.right))

def answer_lca_recursive(root):
    return subtree_result_recursive(root)[1]

def answer_deepest_count(root):
    if root is None:return 0
    q=deque([root]);count=0
    while q:
        count=len(q)
        for _ in range(count):
            node=q.popleft()
            if node.left:q.append(node.left)
            if node.right:q.append(node.right)
    return count

assert answer_height(build_tree([1,2,3,4]))==3
assert answer_lca_recursive(build_tree([1,2,3])).val==1
assert answer_deepest_count(build_tree([1,2,3,4,5]))==2
print('Đáp án mẫu: PASS')

Đáp án mẫu: PASS


## 20. Checklist và template nhớ nhanh

- [ ] Tôi nhận diện được tất cả **lá sâu nhất**, không chỉ lá đầu tiên.
- [ ] Tôi hiểu LCA của một nút là **chính nó**.
- [ ] Tôi phân biệt depth từ gốc và height của cây con.
- [ ] Tôi nhớ `(0,None)` là base case, lá trả `(1, chính_lá)`.
- [ ] Nếu hai con cao bằng nhau, chọn **node hiện tại**; bên nào cao hơn chọn ứng viên của bên đó.
- [ ] Tôi trả `TreeNode` đúng chữ ký và biết xử lý cây lệch sâu.

**Template tư duy:** “DFS mỗi cây con trả `(height,LCA)` → so hai height → chọn candidate → cộng 1 cho height.”